# EXP-02: Training TransH — Semua Kondisi Eksperimen

Notebook ini melatih model **TransH Knowledge Graph Embedding** untuk semua kondisi eksperimen
menggunakan hyperparameter yang **identik** agar perbedaan hasil murni berasal dari kualitas data.

## Hyperparameter (Tetap Sama untuk Semua Kondisi)

| Parameter | Nilai |
|---|---|
| Dimensi Embedding | 192 |
| Learning Rate | ~0.0006 (Adam) |
| Epoch Maksimum | 500 |
| Batch Size | 512 |
| Negative Samples | 50 per triplet positif |
| Evaluasi Validasi | Setiap 10 epoch |
| Early Stopping Patience | 10 evaluasi |
| Random Seed | 42 |

## Cara Penggunaan

1. Set `ACTIVE_CONDITION` ke kondisi yang ingin dilatih
2. **Run All** untuk memulai training
3. Model terbaik disimpan di `Pipeline_Experiments/<CONDITION>/model/`

> **Pastikan** notebook `EXP_00_Preprocessing_Pipeline.ipynb` sudah dijalankan untuk kondisi yang dipilih.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# KONFIGURASI TRAINING
# ══════════════════════════════════════════════════════════════════════════

# Pilih kondisi yang akan dilatih
ACTIVE_CONDITION = "C1_FULL_CLEAN"

# Set True untuk melatih SEMUA kondisi secara berurutan
TRAIN_ALL_CONDITIONS = False

# ── Hyperparameter Model (IDENTIK untuk semua kondisi) ────────────────────
DIMS        = 192      # Dimensi embedding
MARGIN      = 0.507    # Margin ranking loss (gamma)
LR          = 0.000608 # Learning rate Adam
NEG_K       = 50       # Sampel negatif per triplet positif
EPOCHS      = 500      # Epoch maksimum
BATCH_SIZE  = 512      # Ukuran batch
EVAL_EVERY  = 10       # Evaluasi validasi setiap N epoch
PATIENCE    = 10       # Early stopping: N evaluasi tanpa kenaikan
C_SOFT      = 0.001    # Bobot soft constraints
RANDOM_SEED = 42

ALL_CONDITIONS = ["C1_FULL_CLEAN", "C2_NO_TAG_YEAR", "C3_NO_FILTER", "C4_ONLY_USER_FILTER"]

print("=" * 60)
print("TRAINING TRANSH — EKSPERIMEN PREPROCESSING")
print("=" * 60)
if TRAIN_ALL_CONDITIONS:
    print(f"Mode: Latih SEMUA {len(ALL_CONDITIONS)} kondisi")
else:
    print(f"Mode: Kondisi Tunggal → {ACTIVE_CONDITION}")
print(f"\nHyperparameter:")
print(f"  DIMS={DIMS} | MARGIN={MARGIN} | LR={LR}")
print(f"  NEG_K={NEG_K} | EPOCHS={EPOCHS} | BATCH_SIZE={BATCH_SIZE}")
print(f"  EVAL_EVERY={EVAL_EVERY} | PATIENCE={PATIENCE} | SEED={RANDOM_SEED}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# IMPORTS
# ══════════════════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import json
import random
import time
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

BASE_DIR = Path().resolve()
EXP_BASE_DIR = BASE_DIR / "Pipeline_Experiments"

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Seed
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

print(f"\n✅ Imports berhasil.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# ARSITEKTUR MODEL TRANSH
# ══════════════════════════════════════════════════════════════════════════

class TransH(nn.Module):
    """TransH Knowledge Graph Embedding dengan Hyperplane Projection."""
    
    def __init__(self, n_entities: int, n_relations: int, dim: int):
        super().__init__()
        self.dim = dim
        
        # Matriks embedding utama
        self.ent_emb  = nn.Embedding(n_entities,  dim)  # Embedding entitas
        self.rel_emb  = nn.Embedding(n_relations, dim)  # Vektor translasi relasi (d_r)
        self.norm_emb = nn.Embedding(n_relations, dim)  # Vektor normal hyperplane (w_r)
        
        # Inisialisasi uniform
        nn.init.uniform_(self.ent_emb.weight,  -6/dim**0.5, 6/dim**0.5)
        nn.init.uniform_(self.rel_emb.weight,  -6/dim**0.5, 6/dim**0.5)
        nn.init.uniform_(self.norm_emb.weight, -6/dim**0.5, 6/dim**0.5)
        
        # Normalisasi awal
        with torch.no_grad():
            self.ent_emb.weight.data  = torch.nn.functional.normalize(self.ent_emb.weight.data)
            self.rel_emb.weight.data  = torch.nn.functional.normalize(self.rel_emb.weight.data)
            self.norm_emb.weight.data = torch.nn.functional.normalize(self.norm_emb.weight.data)
    
    def _project(self, e, w):
        """Proyeksi entitas e ke hyperplane relasi dengan normal w."""
        # e_perp = e - (e · w) * w
        return e - (e * w).sum(dim=-1, keepdim=True) * w
    
    def score(self, h_idx, r_idx, t_idx):
        """Hitung jarak L2 TransH: ||h_perp + d_r - t_perp||_2"""
        h = self.ent_emb(h_idx)     # (batch, dim)
        r = self.rel_emb(r_idx)     # (batch, dim)
        w = torch.nn.functional.normalize(self.norm_emb(r_idx), dim=-1)  # (batch, dim)
        t = self.ent_emb(t_idx)     # (batch, dim)
        
        h_p = self._project(h, w)   # proyeksi kepala
        t_p = self._project(t, w)   # proyeksi ekor
        
        return torch.norm(h_p + r - t_p, p=2, dim=-1)  # (batch,)
    
    def soft_constraints(self, h_ids, r_ids, t_ids):
        """Hitung soft constraints batch-wise untuk regularisasi.
        
        Hanya menghitung regulasi untuk entitas dan relasi yang aktif
        di dalam batch saat ini, bukan seluruh matriks embedding.
        Ini jauh lebih efisien secara komputasi.
        """
        # 1. Ortogonalitas: (d_r · w_r)^2 / ||d_r||^2 ≈ 0
        d = self.rel_emb(r_ids)
        w = torch.nn.functional.normalize(self.norm_emb(r_ids), dim=-1)
        orth_loss = ((d * w).sum(dim=-1)**2 / (d.norm(dim=-1)**2 + 1e-8)).mean()
        
        # 2. Skala entitas: max(0, ||e||^2 - 1) ≈ 0
        h = self.ent_emb(h_ids)
        t = self.ent_emb(t_ids)
        scale_loss = (torch.clamp(torch.norm(h, dim=-1) - 1, min=0) ** 2 +
                      torch.clamp(torch.norm(t, dim=-1) - 1, min=0) ** 2).mean()
        
        return orth_loss + scale_loss


print("✅ Arsitektur model TransH berhasil didefinisikan.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# NEGATIVE SAMPLER BERBASIS DISTRIBUSI BERNOULLI
# ══════════════════════════════════════════════════════════════════════════

def build_bernoulli_probs(triplets, n_relations):
    """
    Hitung probabilitas korupsi kepala (p_head) per relasi
    berdasarkan rasio TPH (Tails per Head) vs HPT (Heads per Tail).
    """
    from collections import defaultdict
    
    tph_map = defaultdict(lambda: defaultdict(set))  # rel → head → set of tails
    hpt_map = defaultdict(lambda: defaultdict(set))  # rel → tail → set of heads
    
    for h, r, t in triplets:
        tph_map[r][h].add(t)
        hpt_map[r][t].add(h)
    
    p_head = np.zeros(n_relations)
    for r in range(n_relations):
        if tph_map[r]:
            tph = np.mean([len(v) for v in tph_map[r].values()])
        else:
            tph = 1.0
        if hpt_map[r]:
            hpt = np.mean([len(v) for v in hpt_map[r].values()])
        else:
            hpt = 1.0
        p_head[r] = tph / (tph + hpt)
    
    return p_head


def generate_negatives(batch_h, batch_r, batch_t, n_entities, p_head_arr,
                       liked_rel_id, user_ids_set, movie_ids_set, all_triplets_set, neg_k=50):
    """
    Hasilkan sampel negatif per triplet positif (Filtered Negative Sampling).
    Untuk relasi 'liked', batasi korupsi sesuai domain (user atau film).
    """
    batch_size = len(batch_h)
    user_list   = list(user_ids_set)
    movie_list  = list(movie_ids_set)
    
    neg_h_list, neg_r_list, neg_t_list = [], [], []
    
    for i in range(batch_size):
        h, r, t = int(batch_h[i]), int(batch_r[i]), int(batch_t[i])
        ph = float(p_head_arr[r])
        
        for _ in range(neg_k):
            attempts = 0
            while attempts < 100:
                attempts += 1
                if random.random() < ph:
                    # Korupsi kepala
                    if r == liked_rel_id:
                        neg_h = random.choice(user_list)  # ganti dengan user lain
                    else:
                        neg_h = random.randint(0, n_entities - 1)
                    neg_t = t
                else:
                    # Korupsi ekor
                    if r == liked_rel_id:
                        neg_t = random.choice(movie_list)  # ganti dengan film lain
                    else:
                        neg_t = random.randint(0, n_entities - 1)
                    neg_h = h
                
                # Filter agar sampel negatif tidak berupa fakta positif asli
                if (neg_h, r, neg_t) not in all_triplets_set:
                    break
            else:
                # Fallback jika tidak menemukan negatif yang valid setelah 100 percobaan
                if r == liked_rel_id:
                    neg_h = random.choice(user_list)
                    neg_t = random.choice(movie_list)
                else:
                    neg_h = random.randint(0, n_entities - 1)
                    neg_t = random.randint(0, n_entities - 1)
                    
            neg_h_list.append(neg_h)
            neg_r_list.append(r)
            neg_t_list.append(neg_t)
    
    return (
        torch.tensor(neg_h_list, dtype=torch.long),
        torch.tensor(neg_r_list, dtype=torch.long),
        torch.tensor(neg_t_list, dtype=torch.long),
    )


print("✅ Negative sampler Bernoulli berhasil didefinisikan.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FUNGSI EVALUASI (FILTERED SETTING)
# ══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_filtered(model, eval_triplets, all_triplets_set,
                      entity2id, relation2id, movie_ids_list, device,
                      liked_rel="liked", batch_size=64):
    """
    Evaluasi Filtered Setting:
    - Prediksi peringkat film target untuk setiap (User, liked, Movie_GT)
    - Mask semua film yang disukai user di split lain (agar tidak merusak ranking)
    """
    model.eval()
    liked_id = relation2id[liked_rel]
    movie_idx_tensor = torch.tensor(movie_ids_list, dtype=torch.long, device=device)
    movie_idx_map = {mid: idx for idx, mid in enumerate(movie_ids_list)}
    
    # Bangun filter set: user → set movie yang disukai di SEMUA split
    user_all_liked = defaultdict(set)
    for h, r, t in all_triplets_set:
        if r == liked_id:
            user_all_liked[h].add(t)
    
    ranks = []
    
    for h, r, t_gt in eval_triplets:
        if r != liked_id:
            continue
        
        n_movies = len(movie_ids_list)
        h_tensor = torch.full((n_movies,), h, dtype=torch.long, device=device)
        r_tensor = torch.full((n_movies,), r, dtype=torch.long, device=device)
        
        # Hitung skor jarak semua film (pindahkan ke CPU numpy untuk performa cepat)
        scores = model.score(h_tensor, r_tensor, movie_idx_tensor).cpu().numpy()
        
        # Masking film lain yang disukai user (kecuali target) menggunakan O(1) lookup map
        MASK_VALUE = 1e9
        for other_t in user_all_liked[h]:
            if other_t != t_gt and other_t in movie_idx_map:
                scores[movie_idx_map[other_t]] = MASK_VALUE
        
        # Hitung peringkat film target
        target_pos = movie_idx_map[t_gt]
        target_score = scores[target_pos]
        rank = int((scores <= target_score).sum())  # skor kecil = peringkat tinggi
        ranks.append(rank)
    
    if not ranks:
        return {"MRR": 0, "Hits@1": 0, "Hits@3": 0, "Hits@10": 0, "MR": 0}
    
    ranks_arr = np.array(ranks)
    return {
        "MRR":    float(np.mean(1.0 / ranks_arr)),
        "Hits@1":  float(np.mean(ranks_arr <= 1)),
        "Hits@3":  float(np.mean(ranks_arr <= 3)),
        "Hits@10": float(np.mean(ranks_arr <= 10)),
        "MR":     float(np.mean(ranks_arr)),
    }


print("✅ Fungsi evaluasi filtered berhasil didefinisikan.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# FUNGSI UTAMA: train_condition(condition_name)
# ══════════════════════════════════════════════════════════════════════════

def train_condition(condition_name: str):
    """Latih model TransH untuk satu kondisi eksperimen."""
    
    DATA_DIR  = EXP_BASE_DIR / condition_name
    MODEL_DIR = DATA_DIR / "model"
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    
    print("\n" + "═" * 60)
    print(f"  TRAINING: {condition_name}")
    print("═" * 60)
    
    # ── Load data ──────────────────────────────────────────────────────
    print("\n[1] Memuat data...")
    
    def load_tsv(path):
        df = pd.read_csv(path, sep='\t', header=None, names=['h','r','t'])
        return df[['h','r','t']].values.tolist()
    
    train_triplets = load_tsv(DATA_DIR / "train_triplets_int.tsv")
    valid_triplets = load_tsv(DATA_DIR / "valid_triplets_int.tsv")
    test_triplets  = load_tsv(DATA_DIR / "test_triplets_int.tsv")
    
    with open(DATA_DIR / "entity2id.json", 'r') as f:
        entity2id = json.load(f)
    with open(DATA_DIR / "relation2id.json", 'r') as f:
        relation2id = json.load(f)
    
    n_entities  = len(entity2id)
    n_relations = len(relation2id)
    liked_id    = relation2id.get("liked", 5)  # ID relasi 'liked'
    
    # Identifikasi set user dan film
    id2entity    = {v: k for k, v in entity2id.items()}
    user_ids_set = {entity2id[k] for k in entity2id if k.startswith("U_")}
    # Perbaikan: movie_ids_set hanya diisi film asli (sXXXX) agar sampel negatif relevan
    movie_ids_set = {entity2id[k] for k in entity2id if k.startswith("s") and k[1:].isdigit()}
    
    # Cari entitas yang merupakan film asli (bukan user/genre/aktor dll)
    # Film = show_id yang berawal dengan 's' dan diikuti digit (format Netflix)
    movie_ids_list = sorted([entity2id[k] for k in entity2id if k.startswith("s") and k[1:].isdigit()])
    
    print(f"  Entitas    : {n_entities:,}")
    print(f"  Relasi     : {n_relations}")
    print(f"  Train      : {len(train_triplets):,} triplet")
    print(f"  Valid      : {len(valid_triplets):,} triplet")
    print(f"  Film (sXX) : {len(movie_ids_list):,} film")
    print(f"  liked_id   : {liked_id}")
    
    # ── Bangun Bernoulli probs ─────────────────────────────────────────
    print("\n[2] Menghitung probabilitas Bernoulli...")
    p_head = build_bernoulli_probs(train_triplets, n_relations)
    print(f"  p_head per relasi: {[f'{p:.3f}' for p in p_head]}")
    
    # ── Inisialisasi model ─────────────────────────────────────────────
    print("\n[3] Inisialisasi model TransH...")
    model = TransH(n_entities, n_relations, DIMS).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * 0.01)
    margin_loss_fn = nn.MarginRankingLoss(margin=MARGIN)
    
    # Data latih ke tensor
    train_arr = np.array(train_triplets)
    
    # Perbaikan: Menggunakan set (bukan list) untuk lookup O(1) di negative sampling & evaluasi
    all_triplets_set = {(h, r, t) for h, r, t in train_triplets + valid_triplets + test_triplets}
    
    # ── Training loop ──────────────────────────────────────────────────
    print("\n[4] Mulai training...")
    
    best_mrr = 0.0
    best_epoch = 0
    patience_count = 0
    history = {"epoch": [], "loss": [], "valid_mrr": [], "valid_hits10": []}
    
    total_start = time.time()
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        
        # Shuffle data
        perm = np.random.permutation(len(train_arr))
        train_shuffled = train_arr[perm]
        
        epoch_loss = 0.0
        n_batches = 0
        
        for start in range(0, len(train_shuffled), BATCH_SIZE):
            batch = train_shuffled[start:start + BATCH_SIZE]
            bh = torch.tensor(batch[:, 0], dtype=torch.long, device=DEVICE)
            br = torch.tensor(batch[:, 1], dtype=torch.long, device=DEVICE)
            bt = torch.tensor(batch[:, 2], dtype=torch.long, device=DEVICE)
            
            # Negative sampling (dengan menyertakan all_triplets_set untuk Filtered Setting)
            nh, nr, nt = generate_negatives(
                bh.cpu().numpy(), br.cpu().numpy(), bt.cpu().numpy(),
                n_entities, p_head, liked_id, user_ids_set, movie_ids_set, all_triplets_set, NEG_K
            )
            nh = nh.to(DEVICE); nr = nr.to(DEVICE); nt = nt.to(DEVICE)
            
            # Skor positif dan negatif
            # Untuk setiap triplet positif ada NEG_K negatif
            bh_exp = bh.repeat_interleave(NEG_K)
            br_exp = br.repeat_interleave(NEG_K)
            bt_exp = bt.repeat_interleave(NEG_K)
            
            pos_score = model.score(bh_exp, br_exp, bt_exp)
            neg_score = model.score(nh, nr, nt)
            
            # Margin loss: triplet negatif harus lebih tinggi dari positif
            target = -torch.ones_like(pos_score)
            loss = margin_loss_fn(pos_score, neg_score, target)
            
            # Soft constraints (batch-wise: hanya entitas aktif di batch)
            loss = loss + C_SOFT * model.soft_constraints(bh, br, bt)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        scheduler.step()
        
        # Evaluasi validasi setiap EVAL_EVERY epoch
        if epoch % EVAL_EVERY == 0:
            valid_metrics = evaluate_filtered(
                model, valid_triplets, all_triplets_set,
                entity2id, relation2id, movie_ids_list, DEVICE
            )
            mrr = valid_metrics["MRR"]
            h10 = valid_metrics["Hits@10"]
            elapsed = time.time() - total_start
            
            history["epoch"].append(epoch)
            history["loss"].append(avg_loss)
            history["valid_mrr"].append(mrr)
            history["valid_hits10"].append(h10)
            
            improved = "" 
            if mrr > best_mrr:
                best_mrr = mrr
                best_epoch = epoch
                patience_count = 0
                improved = " ⭐ BEST"
                # Simpan model terbaik
                torch.save({
                    "epoch": epoch, "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_mrr": best_mrr,
                    "valid_metrics": valid_metrics,
                    "condition": condition_name,
                    "hyperparams": {"DIMS": DIMS, "MARGIN": MARGIN, "LR": LR,
                                    "NEG_K": NEG_K, "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE}
                }, MODEL_DIR / "best_transh_model.pth")
            else:
                patience_count += 1
            
            print(f"  Epoch {epoch:4d}/{EPOCHS} | Loss: {avg_loss:.4f} | "
                  f"Valid MRR: {mrr:.4f} | Hits@10: {h10:.4f} | "
                  f"[{elapsed:.0f}s]{improved}")
            
            # Early stopping
            if patience_count >= PATIENCE:
                print(f"\n  ⏹ Early Stopping di epoch {epoch} (best={best_epoch}, MRR={best_mrr:.4f})")
                break
        else:
            if epoch % 50 == 0:
                print(f"  Epoch {epoch:4d}/{EPOCHS} | Loss: {avg_loss:.4f}")
    
    total_time = time.time() - total_start
    print(f"\n  Training selesai dalam {total_time/60:.1f} menit")
    print(f"  Best Epoch: {best_epoch} | Best Valid MRR: {best_mrr:.4f}")
    
    # Simpan history
    history_df = pd.DataFrame(history)
    history_df.to_csv(MODEL_DIR / "training_history.csv", index=False)
    
    # Plot kurva training
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Kurva Training — {condition_name}", fontweight='bold')
    
    axes[0].plot(history["epoch"], history["loss"], color='#C44E52', linewidth=2)
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    
    axes[1].plot(history["epoch"], history["valid_mrr"],   color='#4C72B0', linewidth=2, label="MRR")
    axes[1].plot(history["epoch"], history["valid_hits10"], color='#55A868', linewidth=2, label="Hits@10", linestyle='--')
    axes[1].axvline(x=best_epoch, color='gray', linestyle=':', linewidth=1.5, label=f"Best: Epoch {best_epoch}")
    axes[1].set_title("Validation MRR & Hits@10")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Score")
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"  ✅ Model & history disimpan ke: {MODEL_DIR}")
    return best_mrr, best_epoch


print("✅ Fungsi train_condition() berhasil didefinisikan.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# EKSEKUSI TRAINING
# ══════════════════════════════════════════════════════════════════════════

training_results = {}

if TRAIN_ALL_CONDITIONS:
    print("🚀 Melatih SEMUA kondisi secara berurutan...")
    print("⚠ Proses ini akan membutuhkan waktu sangat lama (est. 2-4 jam per kondisi)")
    for condition in ALL_CONDITIONS:
        data_path = EXP_BASE_DIR / condition / "train_triplets_int.tsv"
        if not data_path.exists():
            print(f"⚠ Data untuk {condition} belum ada. Skip.")
            continue
        best_mrr, best_epoch = train_condition(condition)
        training_results[condition] = {"best_mrr": best_mrr, "best_epoch": best_epoch}
    print("\n🎉 SEMUA TRAINING SELESAI!")
else:
    data_path = EXP_BASE_DIR / ACTIVE_CONDITION / "train_triplets_int.tsv"
    if not data_path.exists():
        print(f"❌ ERROR: Data untuk {ACTIVE_CONDITION} tidak ditemukan.")
        print(f"   Pastikan sudah menjalankan EXP_00_Preprocessing_Pipeline.ipynb terlebih dahulu.")
    else:
        best_mrr, best_epoch = train_condition(ACTIVE_CONDITION)
        training_results[ACTIVE_CONDITION] = {"best_mrr": best_mrr, "best_epoch": best_epoch}
        print(f"\n🎉 Training {ACTIVE_CONDITION} SELESAI! Best MRR={best_mrr:.4f} @ epoch {best_epoch}")

# Tampilkan ringkasan
if training_results:
    print("\n=" * 50)
    print("RINGKASAN TRAINING")
    print("=" * 50)
    for cond, res in training_results.items():
        print(f"  {cond:<25} | Best MRR: {res['best_mrr']:.4f} | Best Epoch: {res['best_epoch']}")